In [1]:
from data.update import updateD, updateF
from data.data import D
from data.handler import Handler
from model.gbdt import LGBModel
import pandas as pd

In [3]:
updateD(instruments="csi300")

In [4]:
updateF(instruments="csi300")

In [6]:
# 前复权的数据

# df = D.features(instruments=["sh601212", "sh601033"], fields="all", start_time="2020-01-01", end_time="2025-08-11")

# print(df)

df = D.features(instruments=["sh601033"], fields=["收盘", "开盘"], start_time="2025-08-01", end_time="2025-08-10")

print(df)

# df = D.features(instruments="csi300", fields=["收盘", "开盘", "涨跌幅"], start_time="2025-08-01", end_time="2025-08-11")

# print(df)

                        收盘     开盘
股票代码     日期                      
sh601033 2025-08-01  15.28  15.17
         2025-08-04  15.37  15.21
         2025-08-05  15.41  15.38
         2025-08-06  15.41  15.45
         2025-08-07  15.29  15.41
         2025-08-08  15.31  15.26


In [5]:
df = D.factors(instruments=["sh601033"], fields="all", start_time="2025-08-01", end_time="2025-08-10")

print(df)

                         KMID      KLEN     KMID2       KUP      KUP2  \
股票代码     日期                                                             
sh601033 2025-08-01  0.007251  0.009229  0.785714  0.000000  0.000000   
         2025-08-04  0.010519  0.011834  0.888889  0.000657  0.055556   
         2025-08-05  0.001951  0.007152  0.272727  0.000650  0.090909   
         2025-08-06 -0.002589  0.006472 -0.400000  0.001942  0.300000   
         2025-08-07 -0.007787  0.011032 -0.705882  0.002596  0.235294   
         2025-08-08  0.003277  0.005898  0.555556  0.001966  0.333333   

                         KLOW     KLOW2      KSFT     KSFT2     OPEN0  ...  \
股票代码     日期                                                            ...   
sh601033 2025-08-01  0.001978  0.214286  0.009229  1.000000  0.992801  ...   
         2025-08-04  0.000657  0.055556  0.010519  0.888889  0.989590  ...   
         2025-08-05  0.004551  0.636364  0.005852  0.818182  0.998053  ...   
         2025-08-06  0.00

In [2]:
handler = Handler(
	instruments="csi300", 
    segments={"train": ("2018-01-01", "2023-12-31"),"valid": ("2024-01-01", "2024-12-31"),"test": ("2025-01-01", "2025-12-31")},
	label="[收盘].shift(-1) / [收盘] - 1"
)

model = LGBModel(
    loss="mse",
    colsample_bytree=0.8879,
    learning_rate=0.0421,
    subsample=0.8789,
    lambda_l1=205.6999,
    lambda_l2=580.9768,
    max_depth=8,
    num_leaves=210,
    num_threads=20,
    keep_training_booster=True
)

In [2]:
handler = Handler(
	instruments="csi300", 
    segments={"train": ("2018-01-01", "2023-12-31"),"valid": ("2024-01-01", "2024-12-31"),"test": ("2025-01-01", "2025-12-31")},
	label="[收盘].shift(-1) / [收盘] - 1"
)

df = handler.prepare(key="train", col_set=["raw", "feature", "label"], data_key=Handler.DK_L)

x, y = df["feature"], df["label"]

In [10]:
model.fit(handler, verbose_eval=20)

Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.992586	valid's l2: 0.993741
[40]	train's l2: 0.987761	valid's l2: 0.990419
[60]	train's l2: 0.983692	valid's l2: 0.988604
[80]	train's l2: 0.980283	valid's l2: 0.98747
[100]	train's l2: 0.977195	valid's l2: 0.986729
[120]	train's l2: 0.974233	valid's l2: 0.985966
[140]	train's l2: 0.971419	valid's l2: 0.985801
[160]	train's l2: 0.968809	valid's l2: 0.985643
[180]	train's l2: 0.966286	valid's l2: 0.98578
[200]	train's l2: 0.963867	valid's l2: 0.985945
Early stopping, best iteration is:
[167]	train's l2: 0.967933	valid's l2: 0.985631


In [11]:
model.update_params(colsample_bytree=0.8879,
    learning_rate=0.0421,
    subsample=0.8789,
    lambda_l1=205.6999,
    lambda_l2=580.9768,
    max_depth=8,
    num_leaves=210,
    num_threads=20,)

model.fit(handler, verbose_eval=20)

Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.992586	valid's l2: 0.993741
[40]	train's l2: 0.987761	valid's l2: 0.990419
[60]	train's l2: 0.983692	valid's l2: 0.988604
[80]	train's l2: 0.980283	valid's l2: 0.98747
[100]	train's l2: 0.977195	valid's l2: 0.986729
[120]	train's l2: 0.974233	valid's l2: 0.985966
[140]	train's l2: 0.971419	valid's l2: 0.985801
[160]	train's l2: 0.968809	valid's l2: 0.985643
[180]	train's l2: 0.966286	valid's l2: 0.98578
[200]	train's l2: 0.963867	valid's l2: 0.985945
Early stopping, best iteration is:
[167]	train's l2: 0.967933	valid's l2: 0.985631
